In [31]:
import pandas as pd

# 1. Load the datasets
df1 = pd.read_csv('api_data_aadhar_enrolment_0_500000.csv', parse_dates=['date'], dayfirst=True)
df2 = pd.read_csv('api_data_aadhar_enrolment_500000_1000000.csv', parse_dates=['date'], dayfirst=True)
df3 = pd.read_csv('api_data_aadhar_enrolment_1000000_1006029.csv', parse_dates=['date'], dayfirst=True)

# Print shapes to see initial sizes
print(f"Dataset 1 Shape: {df1.shape}")
print(f"Dataset 2 Shape: {df2.shape}")
print(f"Dataset 3 Shape: {df3.shape}")

# 3. Concatenate (Merge Vertical)
df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)

# Preview the data
df.sample(10)

# 4. Verification
print(f"Combined Dataset Shape: {df.shape}")

Dataset 1 Shape: (500000, 7)
Dataset 2 Shape: (500000, 7)
Dataset 3 Shape: (6029, 7)
Combined Dataset Shape: (1006029, 7)


In [32]:
df.sample(10)

,date,state,district,pincode,age_0_5,age_5_17,age_18_greater
336303,2025-09-26,Assam,Kokrajhar,783336,1,0,0
956866,2025-12-27,Karnataka,Bangalore,560082,1,0,0
355523,2025-09-29,Andhra Pradesh,Cuddapah,516001,2,0,0
618567,2025-11-05,Rajasthan,Jodhpur,342304,2,0,0
776774,2025-11-16,Gujarat,Vadodara,391110,11,0,0
11490,2025-09-01,Karnataka,Bengaluru,560062,2,1,0
274114,2025-09-20,Uttar Pradesh,Etawah,206127,3,2,0
41324,2025-09-03,Maharashtra,Nagpur,441201,1,0,0
382514,2025-10-15,Jammu and Kashmir,Bandipore,193501,4,2,0
233931,2025-09-17,Tamil Nadu,Tiruchirappalli,620101,0,1,0


In [33]:
df.dtypes

date              datetime64[ns]
state                     object
district                  object
pincode                    int64
age_0_5                    int64
age_5_17                   int64
age_18_greater             int64
dtype: object

In [34]:
# 1. Check for Missing Values
print("--- Missing Values ---")
missing_counts = df.isnull().sum()
print(missing_counts[missing_counts > 0])

--- Missing Values ---
Series([], dtype: int64)


In [35]:
# Age counts cannot be negative. Let's verify.
numeric_cols = ['age_0_5', 'age_5_17', 'age_18_greater']
print("\n--- Logical Checks (Negative Values) ---")
for col in numeric_cols:
    neg_count = (df[col] < 0).sum()
    print(f"{col}: {neg_count} negative rows")


--- Logical Checks (Negative Values) ---
age_0_5: 0 negative rows
age_5_17: 0 negative rows
age_18_greater: 0 negative rows


In [37]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np

# Create 'total_population' (Summing the age groups)
df['total_population'] = df['age_0_5'] + df['age_5_17'] + df['age_18_greater']

# Create Percentage Columns (Handling division by zero safely)
# This normalizes the data so you can compare large districts vs small districts
df['perc_0_5'] = np.where(df['total_population'] > 0, (df['age_0_5'] / df['total_population']), 0)
df['perc_5_17'] = np.where(df['total_population'] > 0, (df['age_5_17'] / df['total_population']), 0)
df['perc_18_plus'] = np.where(df['total_population'] > 0, (df['age_18_greater'] / df['total_population']), 0)

# View the new columns
df[['perc_0_5','perc_5_17','perc_18_plus']].head()

,perc_0_5,perc_5_17,perc_18_plus
0,0.100917,0.559633,0.339450
1,0.162791,0.383721,0.453488
2,0.235772,0.666667,0.097561
3,0.584906,0.273585,0.141509
4,0.274510,0.313725,0.411765
